# 01 Tensor walkthrough
Follow one synthetic PET/CT case from files to network output, printing every shape.
Synthetic data only: nothing here is a result.

In [1]:
import sys, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else '.')
import numpy as np, nibabel as nib, torch
from src.data.synthetic import SynthConfig, write_synthetic_hecktor2026
root = Path(tempfile.mkdtemp()) / 'hecktor2026_training'
write_synthetic_hecktor2026(root, SynthConfig(fov_mm=(96, 96, 96), n_per_centre=1, seed=3))
print(sorted(p.name for p in root.iterdir()))

['CHUM-001', 'CHUP-001', 'HECKTOR_2026_Training_Task.csv', 'HGJ-001', 'HMR-001', 'MDA-001', 'USZ-001']


## 1. Two scans, two grids
Same field of view, different voxel size.

In [2]:
for pid in ['CHUM-001', 'CHUP-001']:
    for tag in ['CT', 'PT']:
        img = nib.load(root / pid / f'{pid}__{tag}.nii.gz')
        print(pid, tag, 'shape', img.shape, 'spacing mm', tuple(round(float(z), 2) for z in img.header.get_zooms()[:3]),
              'extent mm', tuple(round(s * z) for s, z in zip(img.shape, img.header.get_zooms())))

CHUM-001 CT shape (96, 96, 96) spacing mm (1.0, 1.0, 1.0) extent mm (96, 96, 96)
CHUM-001 PT shape (24, 24, 24) spacing mm (4.0, 4.0, 4.0) extent mm (96, 96, 96)
CHUP-001 CT shape (96, 96, 32) spacing mm (1.0, 1.0, 3.0) extent mm (96, 96, 96)
CHUP-001 PT shape (24, 24, 24) spacing mm (4.0, 4.0, 4.0) extent mm (96, 96, 96)


## 2. A physical crop
A 48 mm box: CT voxels = 48 / 1 = 48, PET voxels = 48 / 4 = 12 (CHUM). Both start at the same world position.

In [3]:
from src.data.physical_crop import physical_crop, grid_from_affine
pid = 'CHUM-001'
ct, pet, lab = (nib.as_closest_canonical(nib.load(root / pid / f)) for f in
    [f'{pid}__CT.nii.gz', f'{pid}__PT.nii.gz', f'{pid}.nii.gz'])
vols = {'ct': (np.asarray(ct.dataobj), ct.affine), 'pet': (np.asarray(pet.dataobj), pet.affine), 'label': (np.asarray(lab.dataobj), ct.affine)}
centre = grid_from_affine(ct.shape, ct.affine).voxel_to_world(np.array(ct.shape) / 2)
patches, info = physical_crop(vols, 'ct', (48, 48, 48), centre)
for k, v in patches.items(): print(k, v.shape, 'start mm', np.round(info['start_mm'][k], 1), 'spacing', info['spacing'][k])

C:\Users\0561r\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\jit\_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


ct (48, 48, 48) start mm [ 44.3 107.3 -59.6] spacing [1. 1. 1.]
pet (12, 12, 12) start mm [ 44.3 107.3 -59.6] spacing [4. 4. 4.]
label (48, 48, 48) start mm [ 44.3 107.3 -59.6] spacing [1. 1. 1.]


## 3. Into the network
`ACF` takes CT (B,1,X,Y,Z), PET (B,1,x,y,z), the two conditioning vectors and the spacings.

In [4]:
from src.models.acf import ACF
m = ACF(channels=[8, 16, 32, 64, 128], pet_channels=[8, 16, 32]).eval()
ct_t = torch.from_numpy(patches['ct'][None, None].astype('float32'))[..., :48, :48, :48]
pet_t = torch.from_numpy(patches['pet'][None, None].astype('float32'))
print('CT input', tuple(ct_t.shape), 'PET input', tuple(pet_t.shape))
print(m.level_table((48, 48, 48), tuple(pet_t.shape[2:]), (1, 1, 1), (4, 4, 4)))

CT input (1, 1, 48, 48, 48) PET input (1, 1, 12, 12, 12)
[{'level': 'L1', 'channels': 8, 'ct_shape': (48, 48, 48), 'ct_voxel_mm': (1, 1, 1), 'pet_shape': None, 'pet_voxel_mm': None, 'fusion': 'CT only'}, {'level': 'L2', 'channels': 16, 'ct_shape': (24, 24, 24), 'ct_voxel_mm': (2, 2, 2), 'pet_shape': None, 'pet_voxel_mm': None, 'fusion': 'CT only'}, {'level': 'L3', 'channels': 32, 'ct_shape': (12, 12, 12), 'ct_voxel_mm': (4, 4, 4), 'pet_shape': (12, 12, 12), 'pet_voxel_mm': (4, 4, 4), 'fusion': 'P1+L3'}, {'level': 'L4', 'channels': 64, 'ct_shape': (6, 6, 6), 'ct_voxel_mm': (8, 8, 8), 'pet_shape': (6, 6, 6), 'pet_voxel_mm': (8, 8, 8), 'fusion': 'P2+L4'}, {'level': 'L5', 'channels': 128, 'ct_shape': (3, 3, 3), 'ct_voxel_mm': (16, 16, 16), 'pet_shape': (3, 3, 3), 'pet_voxel_mm': (16, 16, 16), 'fusion': 'P3+L5'}]


## 4. Shapes at every encoder level (hooks on the encoder blocks)

In [5]:
shapes = {}
for name, enc in [('CT', m.ct_enc), ('PET', m.pet_enc)]:
    for i, st in enumerate(enc.stages):
        st.register_forward_hook(lambda mod, inp, out, n=name, i=i: shapes.__setitem__(f'{n} level {i+1}', tuple(out.shape)))
cond = torch.zeros(1, 4)
with torch.no_grad():
    out = m(ct_t, pet_t, cond, cond, (1.0, 1.0, 1.0), (4.0, 4.0, 4.0))
for k, v in shapes.items(): print(f'{k:14s}', v)
print('output (B, K, X, Y, Z):', tuple(out.shape))

CT level 1     (1, 8, 48, 48, 48)
CT level 2     (1, 16, 24, 24, 24)
CT level 3     (1, 32, 12, 12, 12)
CT level 4     (1, 64, 6, 6, 6)
CT level 5     (1, 128, 3, 3, 3)
PET level 1    (1, 8, 12, 12, 12)
PET level 2    (1, 16, 6, 6, 6)
PET level 3    (1, 32, 3, 3, 3)
output (B, K, X, Y, Z): (1, 2, 48, 48, 48)


## 5. FiLM is the identity at initialisation

In [6]:
from src.models.hypernet import FiLMHypernet, film
h = FiLMHypernet(4, [8, 16])
(g0, b0), (g1, b1) = h(torch.randn(2, 4))
print('gamma block 0', g0.shape, 'all ones:', bool((g0 == 1).all()), '| beta all zeros:', bool((b0 == 0).all()))
x = torch.randn(2, 8, 4, 4, 4)
print('film(x) == x:', bool(torch.equal(film(x, g0, b0), x)))

gamma block 0 torch.Size([2, 8]) all ones: True | beta all zeros: True
film(x) == x: True


## 6. Parameter counts

In [7]:
print('small ACF', sum(p.numel() for p in m.parameters()))
print('CT hypernet', sum(p.numel() for p in m.ct_hyper.parameters()), '| PET hypernet', sum(p.numel() for p in m.pet_hyper.parameters()))

small ACF 1584434
CT hypernet 36720 | PET hypernet 11760
